Project: build a rainfall prediction system using Random forest classifier.

Workflow: 1) Data collection 2) EDA 3) Data preprocessing 4) Random Forest Classifier 5) Hyperparameter tuning

# 01 Load dependencies and data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.utils import resample
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pickle

In [ ]:
data = pd.read_csv('rainfall.csv')
data.info()

# 02 Preprocessing

In [ ]:
# remove extra spaces from column names
data.columns = data.columns.str.strip()

In [ ]:
data = data.drop(columns=["day"])

In [ ]:
# check missing values
data.isnull().sum()

In [ ]:
data['winddirection'].unique()

In [ ]:
# handle missing values. Use mode for winddirection and median for windspeed because winddirection is categorical and windspeed is numerical. 
# Median is more robust to outliers than mean, so it's a better choice for filling missing values in windspeed.
data['winddirection'] = data['winddirection'].fillna(data['winddirection'].mode()[0])
data['windspeed'] = data['windspeed'].fillna(data['windspeed'].median())

In [ ]:
data['rainfall'].unique()

In [ ]:
# converting target variable to binary
data['rainfall'] = data['rainfall'].map({'yes': 1, 'no': 0})
data.head()

# 03 Exploratory Data Analysis (EDA)

In [ ]:
data.describe()

In [ ]:
sns.set_style('whitegrid')

## a) Distribution plots (dependedent and independent variables)

In [ ]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(['pressure', 'maxtemp', 'temparature', 'mintemp', 'dewpoint', 'humidity', 'cloud', 'sunshine', 'windspeed'], 1):
    plt.subplot(3, 3, i)
    sns.histplot(data[col], kde=True)
    plt.title(f'Distribution of {col} ({i})')
plt.tight_layout()
plt.show()

Since we are predicting rainfall using Random Forest, standardization is not necessary because Random Forest is not sensitive to the scale of the features. 

In [ ]:
# Distribution of target variable
sns.countplot(x='rainfall', data=data)
plt.title('Distribution of Rainfall')
plt.show()

Noticible class imbalance in the target variable. We will use resampling techniques to address this issue before training our model.

## b) Correlation matrix

In [ ]:
# check correlation between features and target variable
plt.figure(figsize=(10, 8))
sns.heatmap(data.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')

Correlation heatmap shows that maxtemp, temperature, mintemp, and dewpoint display multicollinearity, which can lead to overfitting and reduced model performance. 
To address this, we can drop the maxtemp, mintemp, and dewpoint features, as they are highly correlated with temperature and may not provide additional predictive power.

## c) Understanding Outliers

In [ ]:
plt.figure(figsize=(15, 10))
for i, col in enumerate(['pressure', 'maxtemp', 'temparature', 'mintemp', 'dewpoint', 'humidity', 'cloud', 'sunshine', 'windspeed'], 1):
    plt.subplot(3, 3, i)
    sns.boxplot(x=data[col])
    plt.title(f'Boxplot of {col} ({i})')
plt.tight_layout()
plt.show()

Boxplot shows that there are some outliers in the data, but we will keep them for now as they might be important for the model. 
We can try removing them later if we find that they are affecting the model's performance.

# 04 Data processing

## a) Drop correlated features

In [ ]:
# Drop correlated features (keeping dewpoint)
data = data.drop(columns=['maxtemp', 'temparature', 'mintemp'])

## b) Solving imbalance

### Using resample
To solve class imbalance, we can use resampling techniques. Here, we will use oversampling of the minority class (rainfall = 1) to balance the dataset.
Separate majority and minority classes of target variable

In [ ]:
# Separate majority and minority classes of target variable
df_majority = data[data['rainfall'] == 1]
df_minority = data[data['rainfall'] == 0]

In [ ]:
# downsample majority class to match minority class
df_majority_downsampled = resample(df_majority, 
                                 replace=False,                     # sample without replacement
                                 n_samples=len(df_minority),        # to match minority class
                                 random_state=42)                   # reproducible results
print("Downsampled majority class shape:", df_majority_downsampled.shape, "Minority class shape:", df_minority.shape)
assert len(df_majority_downsampled) == len(df_minority), "Downsampling failed to balance classes"

In [ ]:
df_downsampled = pd.concat([df_majority_downsampled, df_minority])
df_downsampled = df_downsampled.sample(frac=1, random_state=42).reset_index(drop=True)  # shuffle the dataset
print("Combined downsampled dataset shape:", df_downsampled.shape)

In [ ]:
# split data into features and target variable
X = df_downsampled.drop(columns=['rainfall'])
y = df_downsampled['rainfall']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Using SMOTE (Synthetic Minority Over-sampling technique)

In [ ]:
SMOTE = True
if SMOTE:
    from imblearn.over_sampling import SMOTE
    X = data.drop(columns=['rainfall'])
    y = data['rainfall']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    smote = SMOTE(sampling_strategy='auto', random_state=42)
    X_train, y_train = smote.fit_resample(X_train, y_train)

# 05 Model training

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

## a) Hypertuning using GridsearchCV

In [ ]:
param_grid = {
    'n_estimators': [10, 15, 20, 25, 30, 35, 40, 50, 150],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)
print("Best Hyperparameters:", grid_search.best_params_)
best_rf_model = grid_search.best_estimator_

# 06 Model Evaluation

In [ ]:
cv_scores = cross_val_score(best_rf_model, X_train, y_train, cv=5)
print("Cross-validation scores:", cv_scores)
print("Average CV score:", np.mean(cv_scores))

In [ ]:
# Test the best model on the test set
y_test_pred = best_rf_model.predict(X_test)
print("Test Set Accuracy:", accuracy_score(y_test, y_test_pred))
print("Test Set Classification Report:\n", classification_report(y_test, y_test_pred))
print("Test Set Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))

# 07 Prediction on Unknown data

In [ ]:
input_data = np.array([[1015.9, 19.9, 95, 81, 0, 40.0, 13.7]])
input_df = pd.DataFrame(input_data, columns=X_train.columns)
prediction = best_rf_model.predict(input_df)
print("Prediction: ","Rainfall" if prediction[0] == 1 else "No Rainfall")
input_df

In [ ]:
# save the model
save_model = False
if save_model == True:
    model_data = {'model': best_rf_model, 'feature_names': X_train.columns.tolist()}

    with open('rainfall_pred_rf_model.pkl', 'wb') as f:
        pickle.dump(model_data, f)

In [ ]:
# Load the model
load_model = False
if load_model == True:
    with open('rainfall_pred_rf_model.pkl', 'rb') as f:
        loaded_model_data = pickle.load(f)
        loaded_model = loaded_model_data['model']
        feature_names = loaded_model_data['feature_names']

# TO DOs:
1) Try other models like XGBoost, SVM, etc. and compare results
2) PCA for dimensionality reduction and visualization
3) Model selection with hyperparameter tuning for all models and compare results